[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/proyectos-integradores/01-saas-contratos.ipynb)

# Proyecto 1: SaaS de análisis de contratos

Notebook interactivo del proyecto completo. Explora cada capa del sistema:
extracción de datos, búsqueda de similares y API REST.

**Prerequisito:** `pip install anthropic chromadb pydantic`

In [ ]:
import anthropic
import json
from datetime import datetime
from pydantic import BaseModel, Field
from typing import Optional

client = anthropic.Anthropic()
print("Cliente listo")

## 1. Modelos Pydantic

In [ ]:
class ClausulaRiesgo(BaseModel):
    titulo: str
    texto: str
    severidad: str  # alta | media | baja
    recomendacion: str

class AnalisisContrato(BaseModel):
    tipo_contrato: str
    partes: list[str]
    fecha_inicio: Optional[str]
    fecha_fin: Optional[str]
    valor_eur: Optional[float]
    renovacion_automatica: bool
    clausulas_riesgo: list[ClausulaRiesgo]
    clausulas_faltantes: list[str]
    puntuacion_riesgo: int = Field(ge=0, le=100)
    resumen_ejecutivo: str
    procesado_en: str

print("Modelos definidos")

## 2. Contrato de prueba

In [ ]:
CONTRATO_PRUEBA = """
CONTRATO DE PRESTACIÓN DE SERVICIOS DE DESARROLLO DE SOFTWARE

En Madrid, a 15 de abril de 2026

PARTES:
- PROVEEDOR: DevCorp Solutions SL, NIF B-98765432, representada por Juan Pérez
- CLIENTE: MiEmpresa SA, NIF A-12345678, representada por Ana García

OBJETO: El Proveedor desarrollará un sistema de gestión de inventario para el Cliente.

PLAZO: 6 meses desde la firma. Inicio: 1 mayo 2026. Fin previsto: 31 octubre 2026.

PRECIO: 60.000€ + IVA, pagaderos en 3 hitos:
- 20.000€ al inicio
- 20.000€ al 50% de avance
- 20.000€ a la entrega final

PENALIZACIONES: El retraso injustificado superior a 15 días hábiles conlleva
una penalización del 0,5% del valor total por día de retraso, hasta un máximo del 10%.

PROPIEDAD INTELECTUAL: Todo el software desarrollado será propiedad exclusiva del Cliente
una vez completado el pago íntegro.

CONFIDENCIALIDAD: Vigente durante la ejecución del contrato y 2 años posteriores.

LEY APLICABLE: Derecho español. Fuero: Juzgados de Madrid.
"""

print(f"Contrato: {len(CONTRATO_PRUEBA)} caracteres")

## 3. Analizar el contrato con Claude

In [ ]:
SYSTEM_LEGAL = """Eres un asistente legal especializado en análisis de contratos.
Responde SIEMPRE con JSON válido. Nunca inventes datos. 
puntuacion_riesgo: 0 (sin riesgo) a 100 (riesgo crítico)."""

def analizar_contrato(texto: str) -> AnalisisContrato:
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system=SYSTEM_LEGAL,
        messages=[{
            "role": "user",
            "content": f"""Analiza este contrato. JSON con esta estructura exacta:
{{
  "tipo_contrato": "prestacion_servicios|compraventa|arrendamiento|laboral|otro",
  "partes": ["Empresa A", "Empresa B"],
  "fecha_inicio": "YYYY-MM-DD o null",
  "fecha_fin": "YYYY-MM-DD o null",
  "valor_eur": 0.0,
  "renovacion_automatica": false,
  "clausulas_riesgo": [
    {{"titulo": "...", "texto": "...", "severidad": "alta|media|baja", "recomendacion": "..."}}
  ],
  "clausulas_faltantes": ["lista de cláusulas que faltan"],
  "puntuacion_riesgo": 0,
  "resumen_ejecutivo": "2-3 frases",
  "procesado_en": "{datetime.now().isoformat()}"
}}

CONTRATO:
{texto[:8000]}"""
        }]
    )
    
    datos = json.loads(response.content[0].text)
    return AnalisisContrato(**datos)

print("Analizando contrato...")
analisis = analizar_contrato(CONTRATO_PRUEBA)
print(f"✅ Análisis completado")
print(f"Tipo: {analisis.tipo_contrato}")
print(f"Valor: {analisis.valor_eur:,.0f}€")
print(f"Puntuación de riesgo: {analisis.puntuacion_riesgo}/100")
print(f"Cláusulas de riesgo: {len(analisis.clausulas_riesgo)}")
print(f"\nResumen: {analisis.resumen_ejecutivo}")

In [ ]:
# Ver cláusulas de riesgo en detalle
print("CLÁUSULAS DE RIESGO DETECTADAS")
print("=" * 55)
for c in analisis.clausulas_riesgo:
    icono = {"alta": "🔴", "media": "🟡", "baja": "🟢"}.get(c.severidad, "⚪")
    print(f"\n{icono} {c.titulo} [{c.severidad.upper()}]")
    print(f"   Texto: {c.texto[:150]}..." if len(c.texto) > 150 else f"   Texto: {c.texto}")
    print(f"   ✏️  {c.recomendacion}")

if analisis.clausulas_faltantes:
    print(f"\n⚠️  CLÁUSULAS FALTANTES:")
    for cf in analisis.clausulas_faltantes:
        print(f"   - {cf}")

## 4. Base de datos vectorial para contratos similares

In [ ]:
import chromadb
import hashlib

def get_embedding_simple(texto: str) -> list[float]:
    """Embedding basado en hash para demo (en producción: Voyage AI o sentence-transformers)."""
    h = int(hashlib.md5(texto.encode()).hexdigest(), 16)
    return [(h >> i & 0xFF) / 255.0 for i in range(384)]

# Inicializar ChromaDB en memoria
chroma_client = chromadb.Client()
col = chroma_client.get_or_create_collection("contratos_historicos")

# Sembrar con contratos de ejemplo
CONTRATOS_HISTORICOS = [
    {"id": "c001", "tipo": "prestacion_servicios", "valor": 45000, "riesgo": 35, 
     "texto": "Contrato de servicios IT entre empresa tecnológica y cliente retail. Sin cláusula de penalización."},
    {"id": "c002", "tipo": "prestacion_servicios", "valor": 80000, "riesgo": 62,
     "texto": "Desarrollo de software a medida con propiedad intelectual compartida. Renovación automática anual."},
    {"id": "c003", "tipo": "compraventa", "valor": 15000, "riesgo": 15,
     "texto": "Compraventa de licencias de software con garantía de 2 años y soporte incluido."},
]

for c in CONTRATOS_HISTORICOS:
    col.add(
        ids=[c["id"]],
        embeddings=[get_embedding_simple(c["texto"])],
        documents=[c["texto"]],
        metadatas=[{"tipo": c["tipo"], "valor_eur": c["valor"], "puntuacion_riesgo": c["riesgo"]}]
    )

# Buscar similares al contrato analizado
resultados = col.query(
    query_embeddings=[get_embedding_simple(CONTRATO_PRUEBA[:500])],
    n_results=3
)

print("CONTRATOS SIMILARES ENCONTRADOS:")
for i, (doc, meta, dist) in enumerate(zip(
    resultados["documents"][0],
    resultados["metadatas"][0],
    resultados["distances"][0]
)):
    print(f"\n{i+1}. Similitud: {1-dist:.1%} | Tipo: {meta['tipo']} | Valor: {meta['valor_eur']:,}€ | Riesgo: {meta['puntuacion_riesgo']}/100")
    print(f"   {doc}")

## 5. Comparación de costes: Haiku vs Sonnet para análisis legal

In [ ]:
# Comparar calidad Haiku vs Sonnet para análisis de contratos
def analizar_con_modelo(texto: str, modelo: str) -> dict:
    t0 = datetime.now()
    resp = client.messages.create(
        model=modelo,
        max_tokens=400,
        system="Analiza este contrato en 2 frases. Identifica el mayor riesgo.",
        messages=[{"role": "user", "content": texto[:2000]}]
    )
    duracion = (datetime.now() - t0).total_seconds()
    
    precios = {
        "claude-haiku-4-5-20251001": {"input": 0.80, "output": 4.00},
        "claude-sonnet-4-6": {"input": 3.00, "output": 15.00}
    }
    p = precios[modelo]
    coste = (resp.usage.input_tokens * p["input"] + resp.usage.output_tokens * p["output"]) / 1_000_000
    
    return {
        "modelo": modelo.split("-")[1],
        "respuesta": resp.content[0].text,
        "duracion_s": round(duracion, 2),
        "coste_usd": round(coste, 5)
    }

print("Comparando modelos para análisis legal...\n")
for modelo in ["claude-haiku-4-5-20251001", "claude-sonnet-4-6"]:
    r = analizar_con_modelo(CONTRATO_PRUEBA, modelo)
    print(f"{'='*55}")
    print(f"Modelo: {r['modelo']} | {r['duracion_s']}s | ${r['coste_usd']}")
    print(r['respuesta'])

print("\n💡 Conclusión: Sonnet vale la pena para análisis legales críticos.")
print("   Haiku es suficiente para triaje inicial o contratos simples.")

## Resumen del proyecto

| Capa | Tecnología | Función |
|---|---|---|
| Análisis | Claude Sonnet | Extracción estructurada con Pydantic |
| Búsqueda | ChromaDB | Contratos similares por semántica |
| API | FastAPI | Endpoints `/analizar-texto` y `/analizar-pdf` |
| Persistencia | SQLite | Historial de análisis realizados |
| Files API | Anthropic | PDFs sin retransmitir el binario |

**Coste estimado en producción:** ~$0.02-0.05 por contrato analizado con Sonnet.